# 步骤 6：可视化与深入分析

本 notebook 复用 `scripts/visualize_results.py` 中的函数，生成与命令行脚本一致的图表。它只读取已有 CSV、checkpoint 和测试集，不会重新训练模型。

## 1. 初始化路径

如果从项目根目录启动 Jupyter，本单元会直接使用当前目录；如果从 `notebooks/` 目录打开，也会自动回到项目根目录。

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd if (cwd / 'scripts' / 'visualize_results.py').exists() else cwd.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root: {root}')

## 2. 导入可视化函数

In [ ]:
from IPython.display import Image, display
import pandas as pd

from scripts.visualize_results import (
    FIGURES_DIR,
    RESULTS_DIR,
    configure_style,
    plot_ablation_delta,
    plot_best_models,
    plot_complexity,
    plot_metric_trends,
    plot_predictions,
    read_inputs,
    write_manifest,
)

configure_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figures will be saved to: {FIGURES_DIR}')

## 3. 读取结果表

- `results/formal_seed42_all.csv`：50 组正式核心实验
- `results/ablation_seed42_vs_formal_comparison.csv`：16 组消融与原模型对比

In [ ]:
formal, ablation = read_inputs(
    RESULTS_DIR / 'formal_seed42_all.csv',
    RESULTS_DIR / 'ablation_seed42_vs_formal_comparison.csv',
)

print(f'Formal rows: {len(formal)}')
print(f'Ablation rows: {len(ablation)}')
display(formal.head())
display(ablation.head())

## 4. 生成汇总图表

In [ ]:
figure_paths = [
    plot_metric_trends(formal),
    plot_best_models(formal),
    plot_complexity(formal),
    plot_ablation_delta(ablation),
]

for path in figure_paths:
    print(path.relative_to(root))

## 5. 使用已有 checkpoint 生成预测曲线与残差图

默认样本包括：

- ETTh1 h96 PatchTST
- ETTh1 h336 PatchTST
- ETTm1 h96 Autoformer
- ETTm1 h336 Autoformer

In [ ]:
prediction_paths, prediction_summaries = plot_predictions(
    batch_size=32,
    sample_idx=0,
    device='cpu',
)
figure_paths.extend(prediction_paths)
manifest_path = write_manifest(figure_paths, prediction_summaries)

display(pd.DataFrame(prediction_summaries))
print(f'Manifest: {manifest_path.relative_to(root)}')

## 6. 预览核心图表

In [ ]:
preview_files = [
    'formal_metric_trends.png',
    'formal_best_model_by_horizon.png',
    'formal_complexity_tradeoff.png',
    'ablation_delta_mse_pct.png',
]

for name in preview_files:
    path = FIGURES_DIR / name
    print(path.relative_to(root))
    display(Image(filename=str(path)))

## 7. 预览预测与残差图

In [ ]:
preview_files = [
    'prediction_ETTh1_h96_patchtst.png',
    'residual_ETTh1_h96_patchtst.png',
    'prediction_ETTm1_h336_autoformer.png',
    'residual_ETTm1_h336_autoformer.png',
]

for name in preview_files:
    path = FIGURES_DIR / name
    print(path.relative_to(root))
    display(Image(filename=str(path)))

## 8. 后续写报告

图表结论已整理在 `docs/analysis_step6.md`。最终报告可以直接引用 `results/figures/` 下的图片和 `results/figures/prediction_samples_summary.csv` 中的推理 shape 记录。